## Changelog
- parent: 20260508_101229_6d2882b7
- change: two purification moves on top of run 23's quality_size pipeline:
    (a) add `quality_interactions` step (FunctionTransformer wrapping
        add_quality_interactions, drop_originals=True) immediately after
        `quality_size`. Adds 3 columns derived from OverallQual/OverallCond:
          • OverallQual_sq = OverallQual ** 2
          • QualCond       = OverallQual * OverallCond
          • QualCond_gap   = OverallQual - OverallCond
        drop_originals=True removes OverallQual and OverallCond after the
        derived features are computed (quality_size has already used them).
    (b) add `drop_redundant` step right after, dropping columns whose
        signal is now captured elsewhere or that are near-constant:
          • {Exter,Bsmt,Garage}{Qual,Cond} → captured in *_x_<Area> from
            quality_size (e.g. ExterQual is in ExterQual_x_GrLivArea).
          • PoolQC → captured in PoolQC_x_PoolArea (and ~99% NA anyway).
          • Utilities, Street, Condition2 → ~99% mode, zero discriminating
            signal.
- hypothesis: prior 4-way blend with raw quality columns + their
    interactions left two redundancies that hurt different models in
    different ways.
    XGBoost: trees waste split budget choosing between near-collinear
        features (OverallQual vs OverallQual_sq, ExterQual vs
        ExterQual_x_GrLivArea). Removing the raw versions consolidates
        the signal into a single splittable feature per concept.
    Lasso/Ridge: one-hot of {Exter,Bsmt,Garage}{Qual,Cond} balloons the
        feature space with columns whose ordinal information is already
        encoded numerically by AmesNAImputer + then multiplied by area
        in quality_size. Dropping cuts dimensionality without losing
        information.
    KRR: polynomial degree=2 kernel already absorbs the squares and
        pairwise products implicitly; explicit children were redundant
        before, and removing them is cheap (slightly fewer kernel
        features → slightly faster, no quality loss).
    Falsification: compared to run 23 (parent), if removing the raw
    columns causes XGBoost CV to improve and the blend to either
    improve or stay flat, the redundancy hypothesis is correct. If
    Lasso/Ridge degrade noticeably, the *Cond columns carried more
    independent signal than this changelog assumed and they should be
    kept (the *Qual ones are still safe to drop, captured by interactions).


In [ ]:
import sys
import numpy as np
import pandas as pd

from pathlib import Path

IS_KAGGLE = Path("/kaggle/input").exists()

if IS_KAGGLE:
    data_dir   = Path("/kaggle/input/home-data-for-ml-course")
    output_dir = Path("/kaggle/working")
else:
    def _find_competition_dir(start: Path) -> Path:
        for p in [start, *start.parents]:
            if (p / "config.yaml").exists() and (p / "data").is_dir():
                return p
        raise RuntimeError("competition dir not found (no ancestor has config.yaml + data/)")

    comp_dir   = _find_competition_dir(Path.cwd())
    data_dir   = comp_dir / "data"
    output_dir = Path.cwd()

    if str(comp_dir) not in sys.path:
        sys.path.insert(0, str(comp_dir))

train_data_raw = pd.read_csv(data_dir / "train.csv")
test_data_raw  = pd.read_csv(data_dir / "test.csv")

In [ ]:
# see eda-TotalSF.ipynb — drop the mega-house outliers (TotalSF > 7000)
_outlier_mask = (
    train_data_raw["TotalBsmtSF"]
    + train_data_raw["1stFlrSF"]
    + train_data_raw["2ndFlrSF"]
) > 7000
train_data = train_data_raw.loc[~_outlier_mask].reset_index(drop=True)

# Drop Id (row identifier, no signal) and SalePrice (target). Everything else
# goes through preprocessing.
DROP = ["Id", "SalePrice"]
X      = train_data.drop(columns=DROP, errors="ignore").copy()
X_test = test_data_raw.drop(columns=DROP, errors="ignore").copy()
y      = np.log1p(train_data["SalePrice"])

# MSSubClass is a nominal int code — cast to string so the encoder treats it
# as a category rather than an ordered number.
X["MSSubClass"]      = X["MSSubClass"].astype(str)
X_test["MSSubClass"] = X_test["MSSubClass"].astype(str)

# Auto-detect num/cat from train; apply the same split to test.
NUMERIC     = X.select_dtypes(include="number").columns.tolist()
CATEGORICAL = X.select_dtypes(exclude="number").columns.tolist()
print(f"{len(NUMERIC)} numeric + {len(CATEGORICAL)} categorical = {len(X.columns)} total")

In [ ]:
from sklearn.base import BaseEstimator, TransformerMixin, clone
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import FunctionTransformer, RobustScaler
from sklearn.linear_model import Lasso, Ridge
from sklearn.kernel_ridge import KernelRidge
from sklearn.model_selection import KFold, cross_val_score

from xgboost import XGBRegressor

from utils.ames_sklearn_pipeline import AmesNAImputer, AmesEncoder
from utils.ames_feature_engineering import (
    add_size_features, add_temporal_features, add_bath_features,
    add_quality_size_features, add_quality_interactions,
)


# Columns that become redundant after the FE steps above:
#  - {Exter,Bsmt,Garage}{Qual,Cond} are all captured in the corresponding
#    *_x_<Area> interaction features produced by add_quality_size_features.
#  - PoolQC is captured in PoolQC_x_PoolArea (and the column is ~99% NA anyway).
#  - Utilities, Street, Condition2 are ~99% mode and carry zero signal.
REDUNDANT_AFTER_FE = [
    "ExterQual", "ExterCond",
    "BsmtQual",  "BsmtCond",
    "GarageQual", "GarageCond",
    "PoolQC",
    "Utilities",
    "Street", "Condition2",
]


def drop_redundant(df):
    return df.drop(columns=[c for c in REDUNDANT_AFTER_FE if c in df.columns])


class CategoryCaster(BaseEstimator, TransformerMixin):
    """Pin categorical vocabulary at fit-time, replay at transform."""

    def fit(self, X, y=None):
        self.vocab_ = {}
        cat_like = X.select_dtypes(include=["object", "string", "category"]).columns
        for c in cat_like:
            self.vocab_[c] = sorted(X[c].dropna().astype(str).unique())
        return self

    def transform(self, X):
        out = X.copy()
        for c, cats in self.vocab_.items():
            if c in out.columns:
                out[c] = pd.Categorical(out[c].astype(object), categories=cats)
        return out


class BoxCoxSkewed(BaseEstimator, TransformerMixin):
    """boxcox1p(lam) on numeric columns with |skew| > threshold (per-fold)."""

    def __init__(self, threshold: float = 0.75, lam: float = 0.15):
        self.threshold = threshold
        self.lam = lam

    def fit(self, X, y=None):
        from scipy.stats import skew
        numeric = X.select_dtypes(include="number").columns
        skews = X[numeric].apply(lambda s: skew(s.dropna()))
        self.skewed_cols_ = skews[skews.abs() > self.threshold].index.tolist()
        return self

    def transform(self, X):
        from scipy.special import boxcox1p
        out = X.copy()
        for c in self.skewed_cols_:
            if c in out.columns:
                out[c] = boxcox1p(out[c], self.lam)
        return out


# ---- XGBoost pipeline (run 19's best config — native categorical, no scaling)
xgb_pipe = Pipeline([
    ("na",                    AmesNAImputer()),
    ("quality_size",          FunctionTransformer(
                                  add_quality_size_features,
                                  kw_args={"drop_originals": False},
                              )),
    ("quality_interactions",  FunctionTransformer(
                                  add_quality_interactions,
                                  kw_args={"drop_originals": False},
                              )),                       # drops OverallQual/OverallCond
    ("drop_redundant",        FunctionTransformer(drop_redundant)),
    ("size",                  FunctionTransformer(
                                  add_size_features,
                                  kw_args={"drop_originals": True},
                              )),
    ("fe",                    FunctionTransformer(
                                  add_temporal_features,
                                  kw_args={"drop_originals": True},
                              )),
    ("bath",                  FunctionTransformer(
                                  add_bath_features,
                                  kw_args={"drop_originals": False},
                              )),
    ("cat_cast",              CategoryCaster()),
    ("model",                 XGBRegressor(
                                  tree_method="hist",
                                  enable_categorical=True,
                                  learning_rate=0.05,
                                  n_estimators=600,
                                  max_depth=4,
                                  min_child_weight=1,
                                  reg_lambda=1,
                                  subsample=0.8,
                                  colsample_bytree=0.8,
                                  random_state=42,
                                  n_jobs=1,
                                  verbosity=0,
                              )),
])


# ---- Linear/kernel pipeline factory: BoxCox → one-hot → RobustScale → model
def make_linear_pipe(model):
    return Pipeline([
        ("na",                    AmesNAImputer()),
        ("quality_size",          FunctionTransformer(
                                      add_quality_size_features,
                                      kw_args={"drop_originals": False},
                                  )),
        ("quality_interactions",  FunctionTransformer(
                                      add_quality_interactions,
                                      kw_args={"drop_originals": False},
                                  )),                   # drops OverallQual/OverallCond
        ("drop_redundant",        FunctionTransformer(drop_redundant)),
        ("size",                  FunctionTransformer(
                                      add_size_features,
                                      kw_args={"drop_originals": True},
                                  )),
        ("fe",                    FunctionTransformer(
                                      add_temporal_features,
                                      kw_args={"drop_originals": True},
                                  )),
        ("bath",                  FunctionTransformer(
                                      add_bath_features,
                                      kw_args={"drop_originals": False},
                                  )),
        ("skew",                  BoxCoxSkewed(threshold=0.75, lam=0.15)),
        ("encoder",               AmesEncoder()),
        ("scaler",                RobustScaler()),
        ("model",                 model),
    ])


lasso_pipe = make_linear_pipe(
    Lasso(alpha=0.0005, random_state=1, max_iter=10000)
)
ridge_pipe = make_linear_pipe(
    Ridge(alpha=10, random_state=2)
)
krr_pipe = make_linear_pipe(
    KernelRidge(alpha=0.6, kernel="polynomial", degree=2, coef0=2.5)
)


# ---- Per-model CV
cv = KFold(n_splits=5, shuffle=True, random_state=42)

print("Per-model CV RMSE (log-price):")
model_pipes = {
    "XGBoost": xgb_pipe,
    "Lasso":   lasso_pipe,
    "Ridge":   ridge_pipe,
    "KRR":     krr_pipe,
}
for name, pipe in model_pipes.items():
    s = -cross_val_score(pipe, X, y, scoring="neg_root_mean_squared_error", cv=cv, n_jobs=1)
    print(f"  {name:8s}: {s.mean():.4f} ± {s.std():.4f}")


# ---- Blend CV: arithmetic mean of expm1 predictions, RMSE in log-space
print("\nBlend CV (XGBoost + Lasso + Ridge + KRR, equal weights):")
fold_rmses = []
for tr_idx, vl_idx in cv.split(X):
    X_tr, X_vl = X.iloc[tr_idx], X.iloc[vl_idx]
    y_tr, y_vl = y.iloc[tr_idx], y.iloc[vl_idx]
    fold_dollars = []
    for pipe in [xgb_pipe, lasso_pipe, ridge_pipe, krr_pipe]:
        p = clone(pipe)
        p.fit(X_tr, y_tr)
        fold_dollars.append(np.expm1(p.predict(X_vl)))
    blend_dollar = np.mean(fold_dollars, axis=0)
    blend_log    = np.log1p(blend_dollar)
    fold_rmses.append(np.sqrt(np.mean((blend_log - y_vl.values) ** 2)))
fold_rmses = np.array(fold_rmses)
print(f"  Blend  : {fold_rmses.mean():.4f} ± {fold_rmses.std():.4f}")
print(f"  Per-fold: {np.round(fold_rmses, 4).tolist()}")


# ---- Refit each on full data for prediction
for pipe in (xgb_pipe, lasso_pipe, ridge_pipe, krr_pipe):
    pipe.fit(X, y)

In [ ]:
preds_dollars = np.stack([
    np.expm1(xgb_pipe.predict(X_test)),
    np.expm1(lasso_pipe.predict(X_test)),
    np.expm1(ridge_pipe.predict(X_test)),
    np.expm1(krr_pipe.predict(X_test)),
])
test_pred = preds_dollars.mean(axis=0)

sample = pd.read_csv(data_dir / "sample_submission.csv")
submission = sample.copy()
submission["SalePrice"] = test_pred
submission.to_csv(output_dir / "submission.csv", index=False)